## Dataset:

In [1]:
# pandas is used to load and manipulate the dataset as a DataFrame (table-like structure)
import pandas as pd

In [2]:
# Load the IMDB dataset (50,000 movie reviews with sentiment labels) into a DataFrame
df = pd.read_csv('IMDB Dataset.csv')

# Preview the first 5 rows to check the data loaded correctly and see its structure
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
# Check dataset dimensions: (number of rows, number of columns) -> expect (50000, 2)
df.shape

(50000, 2)

### Lowercase

In [ ]:
# Preview a single review (index 2) before applying any preprocessing
# so we have a baseline to compare against after each step
df['review'][2]

In [ ]:
# View the full review column to confirm it loaded correctly for all 50,000 rows
df['review']

### Lowercase

In [6]:
# Convert all review text to lowercase so that words like "Great" and "great"
# are treated as the same token during later steps (tokenization, vectorization, etc.)
df['review'] = df['review'].str.lower()

# Check if method is applied
df['review'][2]

'i thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. the plot is simplistic, but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). while some may be disappointed when they realize this is not match point 2: risk addiction, i thought it was proof that woody allen is still fully in control of the style many of us have grown to love.<br /><br />this was the most i\'d laughed at one of woody\'s comedies in years (dare i say a decade?). while i\'ve never been impressed with scarlet johanson, in this she managed to tone down her "sexy" image and jumped right into a average, but spirited young woman.<br /><br />this may not be the crown jewel of his career, but it was wittier than "devil wears prada" and more interesting than "superman" a great comedy to go see with friends.'

### Remove HTML Tags

In [7]:
# re (regular expressions) lets us define a pattern to find and strip out HTML tags
import re

def remove_html_tags(text):
    # Matches anything between '<' and '>' (e.g. <br />, <div>, </p>) and deletes it
    pattern = re.compile(r'<.*?>')
    return pattern.sub('', text)

# Apply the function to every review in the column, replacing tags like <br /><br /> with nothing
df['review'] = df['review'].apply(remove_html_tags)
df['review'][2]

'i thought this was a wonderful way to spend time on a too hot summer weekend, sitting in the air conditioned theater and watching a light-hearted comedy. the plot is simplistic, but the dialogue is witty and the characters are likable (even the well bread suspected serial killer). while some may be disappointed when they realize this is not match point 2: risk addiction, i thought it was proof that woody allen is still fully in control of the style many of us have grown to love.this was the most i\'d laughed at one of woody\'s comedies in years (dare i say a decade?). while i\'ve never been impressed with scarlet johanson, in this she managed to tone down her "sexy" image and jumped right into a average, but spirited young woman.this may not be the crown jewel of his career, but it was wittier than "devil wears prada" and more interesting than "superman" a great comedy to go see with friends.'

### Remove URL

In [ ]:
def remove_url(text):
    # Matches both http/https URLs and bare www. addresses and removes them
    # Useful for reviews that mention websites, as URLs add no sentiment signal
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

### Punctuation Handling

In [ ]:
# string provides a ready-made set of all punctuation characters
# time is imported for benchmarking preprocessing steps if needed
import string, time
string.punctuation

In [ ]:
# Store all punctuation characters in a variable to reuse in removal functions
exclude = string.punctuation
exclude

In [ ]:
# NOTE: This approach is incorrect — it overwrites 'tect' each iteration
# but never writes back to 'text', so the original string is returned unchanged.
# See remove_punc_1 below for the correct approach using str.translate().
def remove_punc(text):
    for char in text:
        tect = text.replace(char, '')
    return text

In [12]:
def remove_punc_1(text):
    # Use .str accessor for pandas Series, or .translate() for strings
    if isinstance(text, pd.Series):
        return text.str.translate(str.maketrans('', '', exclude))
    else:
        return text.translate(str.maketrans('', '', exclude))

In [ ]:
# Apply punctuation removal to the entire review column and preview the result
# Note: this does not update df['review'] in place — assign back if you want to persist it
remove_punc_1(df['review'])

### Chat Conversion Handling

In [ ]:
# We tend to use short forms or slangs in our chat convos, so we will be handling this
# Very highly used for Social media tasks

# Created a dictionary to reference what short form means what
chat_words = {
    'AFAIK':'As Far As I Know',
    'AFK':'Away From Keyboard',
    'ASAP':'As Soon As Possible',
    "FYI": "For Your Information",
    "ASAP": "As Soon As Possible",
    "BRB": "Be Right Back",
    "BTW": "By The Way",
    "OMG": "Oh My God",
    "IMO": "In My Opinion",
    "LOL": "Laugh Out Loud",
    "TTYL": "Talk To You Later",
    "GTG": "Got To Go",
    "TTYT": "Talk To You Tomorrow",
    "IDK": "I Don't Know",
    "TMI": "Too Much Information",
    "IMHO": "In My Humble Opinion",
    "ICYMI": "In Case You Missed It",
    "AFAIK": "As Far As I Know",
    "BTW": "By The Way",
    "FAQ": "Frequently Asked Questions",
    "TGIF": "Thank God It's Friday",
    "FYA": "For Your Action",
    "ICYMI": "In Case You Missed It",
}

In [ ]:
def chat_convo(text):
    new_text = []
    # Split the text into individual words and check each one against the lookup dict
    for w in text.split():
        # Compare in uppercase so matching is case-insensitive (e.g. "asap" == "ASAP")
        if w.upper() in chat_words:
            new_text.append(chat_words[w.upper()])  # Replace slang with its full form
        else:
            new_text.append(w)  # Keep the word as-is if it's not a known abbreviation
    return " ".join(new_text)  # Reassemble the words back into a single string

In [ ]:
# Test the function: "ASAP" should be expanded to "As Soon As Possible"
chat_convo('Do this work ASAP')

### Incorrect Text Handling

In [ ]:
# TextBlob is a simple NLP library that can detect and auto-correct spelling errors
from textblob import TextBlob

In [ ]:
incorrect_text = 'ceertain conditionas duriing seveal aree'

# Wrap the text in a TextBlob object to access NLP methods
textBlb = TextBlob(incorrect_text)

# .correct() uses word frequency statistics to suggest the most likely correct spelling
# .string extracts the corrected text as a plain Python string
textBlb.correct().string

### Stopwords

In [ ]:
# nltk is the Natural Language Toolkit — provides corpora, tokenizers, and more
import nltk
from nltk.corpus import stopwords

# Download the stopwords corpus (common words like "the", "is", "and") on first run
nltk.download('stopwords')

In [ ]:
# Preview the full English stopwords list — 179 common words that carry little meaning
stopwords.words('english')

In [ ]:
def remove_stopwords(text):
    new_text = []

    for word in text.split():
        if word in stopwords.words('english'):
            new_text.append('')   # Replace stopword with empty string (preserves spacing)
        else:
            new_text.append(word)

    # Copy the list before clearing — clear() would wipe it before join() could use it
    x = new_text[:]
    new_text.clear()
    return " ".join(x)

In [ ]:
# Test: stopwords like "my", "a", "of", "and" should be removed
remove_stopwords('probably my all-time fav movie, a story of selflessness and sacrifice')

In [ ]:
# Inspect the current state of the DataFrame before applying stopword removal
df.head()

In [ ]:
# Do not run this
df['review'].apply(remove_stopwords)

### Emojis

In [ ]:
import re

def remove_emoji(text):
    # Build a pattern covering the main Unicode emoji ranges
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)  # Strip all matched emoji, leaving clean text

In [ ]:
# Test: laughing emojis should be stripped, leaving only the text
remove_emoji("Lmao 😂😂")

In [ ]:
# Test with a sentence to confirm emoji removal doesn't affect surrounding words
remove_emoji("Loved the movie. It was 😘😘")

In [ ]:
# Install emoji into the same Python the kernel uses (avoids environment mismatch)
import sys
!{sys.executable} -m pip install emoji

In [ ]:
# Alternative to removal: demojize() converts emoji to their text shortcodes
# Useful when you want to keep the sentiment signal (e.g. 🔥 → :fire:)
import emoji
print(emoji.demojize('Python is 🔥'))

In [ ]:
# Test demojize on a sentiment-heavy emoji — preserves meaning as readable text
print(emoji.demojize('Loved the movie. It was 😘'))

### Tokenization